# Tutorial 3c: Stepped Optimization and DLS/LM

This tutorial covers the **Levenberg-Marquardt (DLS)** native solver in depth:

* How the LM algorithm works: `(JᵀJ + λI)Δx = -Jᵀr`
* Customising the `LevenbergController` (λ-schedule)
* Using `LineSearchController`
* Exact equality constraints via `NullSpaceStrategy`
* Bounds via `BoxBoundsStrategy` (the default)
* `GaussNewton` for the un-damped variant

**Prerequisite:** Tutorial 3b (method selection). [Tutorial 3d](Tutorial_3d_Inspecting_and_Controlling_a_Run.ipynb) covers observers and composable stopping in depth.

## 1. Setup

In [ ]:
import copy
import numpy as np

from optiland import optic
from optiland.optimization import (
    minimize,
    OptimizationProblem,
    LevenbergController,
    LineSearchController,
    LevenbergMarquardt,
    GaussNewton,
    NullSpaceStrategy,
)


Build a cemented doublet as the working example.

In [ ]:
def make_doublet():
    lens = optic.Optic()
    lens.surfaces.add(index=0, thickness=np.inf)
    lens.surfaces.add(index=1, thickness=6.0, radius=25.0, material='N-BK7', is_stop=True)
    lens.surfaces.add(index=2, thickness=2.0, radius=-18.0, material='N-SF11')
    lens.surfaces.add(index=3, thickness=0.5, radius=-50.0)
    lens.surfaces.add(index=4, thickness=48.0)
    lens.surfaces.add(index=5)
    lens.set_aperture(aperture_type='EPD', value=10)
    lens.fields.set_type('angle')
    lens.fields.add(y=0.0)
    lens.fields.add(y=0.7)
    lens.wavelengths.add(value=0.4861)
    lens.wavelengths.add(value=0.5876, is_primary=True)
    lens.wavelengths.add(value=0.6563)
    lens.update_paraxial()
    return lens

lens = make_doublet()


## 2. The DLS algorithm

Levenberg-Marquardt minimises a sum-of-squares merit function:

```
(JᵀJ + λI)Δx = -Jᵀr
```

Large `λ` gives a small gradient-like step (safe far from the solution); small `λ` gives a fast Gauss-Newton step (efficient near the solution). The `LevenbergController` adapts `λ` automatically.

## 3. Default DLS run

In [ ]:
def make_problem(lens):
    problem = OptimizationProblem()
    for field in lens.fields.get_field_coords():
        input_data = {'optic': lens, 'surface_number': -1,
                      'Hx': field[0], 'Hy': field[1],
                      'num_rays': 5, 'wavelength': 0.5876, 'distribution': 'hexapolar'}
        problem.add_operand('rms_spot_size', target=0, weight=1, input_data=input_data)
    problem.add_operand('effective_focal_length', target=100, weight=1, input_data={'optic': lens})
    for s in [1, 2, 3]:
        problem.add_variable(lens, 'radius', surface_number=s)
    problem.add_variable(lens, 'thickness', surface_number=4, min_val=30, max_val=80)
    return problem

problem = make_problem(lens)
result = minimize(problem, 'dls')
print(result)


## 4. Custom `LevenbergController`

Tune `lam_init` (starting damping), `lam_factor` (adaptation speed), and `lam_max` (safety cap). A smaller `lam_init` starts more aggressively.

In [ ]:
lens2 = make_doublet()
problem2 = make_problem(lens2)

ctrl = LevenbergController(lam_init=1e-4, lam_factor=10.0, lam_max=1e10)
result2 = minimize(problem2, 'dls', controller=ctrl)
print(result2)


## 5. `LineSearchController`

Replaces the λ-damping with a backtracking line search along the Gauss-Newton direction. Useful when you are already near a solution.

In [ ]:
lens3 = make_doublet()
problem3 = make_problem(lens3)

ctrl_ls = LineSearchController()
result3 = minimize(problem3, 'dls', controller=ctrl_ls)
print(result3)


## 6. `GaussNewton` (un-damped)

`method='gauss_newton'` uses un-damped Gauss-Newton steps (`λ = 0`). Converges fast near the solution but can diverge far from it. Useful as a fine-tuning pass after DLS.

In [ ]:
lens4 = make_doublet()
problem4 = make_problem(lens4)
result4 = minimize(problem4, 'gauss_newton')
print(result4)


## 7. Variable bounds (`BoxBoundsStrategy`)

Add `min_val`/`max_val` to `add_variable(...)`. With the default `bounds=True`, the optimizer clips/projects parameters at each step. Set `bounds=False` to disable.

In [ ]:
lens5 = make_doublet()
problem5 = make_problem(lens5)
# radius variables already have bounds in make_problem via thickness only;
# let's add explicit radius bounds
problem5_bounded = OptimizationProblem()
for field in lens5.fields.get_field_coords():
    input_data = {'optic': lens5, 'surface_number': -1,
                  'Hx': field[0], 'Hy': field[1],
                  'num_rays': 5, 'wavelength': 0.5876, 'distribution': 'hexapolar'}
    problem5_bounded.add_operand('rms_spot_size', target=0, weight=1, input_data=input_data)
problem5_bounded.add_operand('effective_focal_length', target=100, weight=1, input_data={'optic': lens5})
for s in [1, 2, 3]:
    problem5_bounded.add_variable(lens5, 'radius', surface_number=s, min_val=10, max_val=200)
problem5_bounded.add_variable(lens5, 'thickness', surface_number=4, min_val=30, max_val=80)
result5 = minimize(problem5_bounded, 'dls', bounds=True)
print(result5)


## 8. `NullSpaceStrategy` for exact equality

`NullSpaceStrategy` enforces equality operands (those with `target=`) by projecting each candidate step into the null space of the equality Jacobian. The constraint is satisfied exactly at every step.

In [ ]:
lens6 = make_doublet()
problem6 = make_problem(lens6)
result6 = minimize(problem6, 'dls', constraints=NullSpaceStrategy())
print(result6)


## 9. Summary

| Feature | How to enable |
|---------|---------------|
| Default DLS | `minimize(problem, 'dls')` |
| Custom λ schedule | `controller=LevenbergController(lam_init=..., lam_factor=...)` |
| Line-search controller | `controller=LineSearchController()` |
| Un-damped Gauss-Newton | `minimize(problem, 'gauss_newton')` |
| Box bounds | `add_variable(..., min_val=..., max_val=...)` + `bounds=True` |
| Exact equality projection | `constraints=NullSpaceStrategy()` |

**Next:** [Tutorial 3d](Tutorial_3d_Inspecting_and_Controlling_a_Run.ipynb) — observers, stopping criteria, and run control.